# Visual Health — localisation des lésions sur IDRiD (Kaggle)

Cartes d'occlusion et tokens de patch de RETFound et RETFound-Green sur les 81
images de la segmentation IDRiD (`scripts/localization_maps.py`). L'analyse
contre les masques d'experts se fait ensuite en local
(`scripts/localization_analysis.py`).

**Avant de lancer** :
1. `Add Input` : dataset `aaryapatel98/indian-diabetic-retinopathy-image-dataset`.
2. `Settings > Accelerator > GPU T4 x2` (pas P100), `Internet > On`.
3. `Add-ons > Secrets` : `HF_TOKEN` coché.
4. **`Save Version > Save & Run All (Commit)`** (~15 min). Le fichier
   `visual_health_localization.zip` apparaît ensuite dans l'onglet `Output`.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : activer GPU T4 x2"
!df -h /kaggle/working /kaggle/temp | tail -2


In [ ]:
import os

OUTPUT = "/kaggle/working"  # seul le zip y est écrit
WORK = "/kaggle/temp"
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
![ -d visual-health-platform ] || git clone -q https://github.com/sumarc183-design/visual-health-platform.git
!cd visual-health-platform && git fetch -q && git reset -q --hard origin/main && git log --oneline -1
%cd {WORK}/visual-health-platform
!pip install -q -r requirements.txt


In [ ]:
import glob

TARGET = "third_party/retfound/checkpoints/RETFound_mae_natureCFP.pth"
os.makedirs(os.path.dirname(TARGET), exist_ok=True)
found = sorted(glob.glob("/kaggle/input/**/RETFound_mae_natureCFP.pth", recursive=True))
if os.path.exists(TARGET):
    print("Poids RETFound déjà présents")
elif found:
    os.symlink(found[0], TARGET)
    print("Poids RETFound pris dans les inputs :", found[0])
else:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")  # jamais écrit dans le notebook
    !python scripts/download_retfound_checkpoint.py --repo-id YukunZhou/RETFound_mae_natureCFP --out third_party/retfound/checkpoints
CHECKPOINT = TARGET

GREEN = "third_party/retfound_green/retfoundgreen_statedict.pth"
if not os.path.exists(GREEN):
    !mkdir -p third_party/retfound_green
    !curl -sL --fail -o {GREEN} https://github.com/justinengelmann/RETFound_Green/releases/download/v0.1/retfoundgreen_statedict.pth
GREEN_SHA = "431de5dbc1bebbb32f60e2c0bcf8daa4f8bcbf06f7cb1e1dc97ec589713942e1"
found = !sha256sum {GREEN}
if found[0].split()[0] != GREEN_SHA:
    raise RuntimeError(f"Poids RETFound-Green inattendus : {found[0]}")
print(CHECKPOINT, GREEN, "OK")


In [ ]:
import glob
import time
import zipfile

hits = glob.glob("/kaggle/input/**/2. All Segmentation Groundtruths", recursive=True)
if not hits:
    raise RuntimeError(f"IDRiD introuvable : ajouter le dataset en input. Inputs : {os.listdir('/kaggle/input')}")
IDRID = os.path.dirname(sorted(hits)[0])
print("IDRiD :", IDRID)

t0 = time.time()
!python -u scripts/localization_maps.py --data-root "{IDRID}" --dr-heads results/localization/dr_heads.npz \
    --retfound-checkpoint {CHECKPOINT} --green-checkpoint {GREEN} --out-dir localization 2>&1 | grep -v -i warning
for f in ("retfound", "green"):
    path = f"localization/localization_{f}.npz"
    if not os.path.exists(path) or os.path.getmtime(path) < t0:
        raise RuntimeError(f"localization_maps.py a échoué : {path} absent (voir la sortie ci-dessus)")

archive = f"{OUTPUT}/visual_health_localization.zip"
with zipfile.ZipFile(archive, "w") as z:
    for f in sorted(glob.glob("localization/*.npz")):
        z.write(f)
print("->", archive, f"({os.path.getsize(archive) / 1e6:.0f} Mo) en {(time.time() - t0) / 60:.0f} min")
